In [1]:
!pip install -q datasets datasketch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 4.2 MB/s eta 0:00:00


In [ ]:
# Sources kept:
#   1. AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.1   core defensive reasoning
#   2. AlicanKiraz0/All-CVE-Records-Training-Dataset     structured vulnerability analysis
#   3. tihanyin/CyberMetric                              factual knowledge, short answers
#   4. luongnv89/phishing-email                          email triage, second task

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from huggingface_hub import notebook_login


In [ ]:
notebook_login()

In [ ]:
import os

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/Aegis-CyberSec-Guard"
DATA_DIR = os.path.join(DRIVE_ROOT, "dataV5")

In [ ]:
TRAIN_PATH = os.path.join(DATA_DIR, "master_train.jsonl")
VAL_PATH = os.path.join(DATA_DIR, "master_val.jsonl")
HOLDOUT_PATH = os.path.join(DATA_DIR, "holdout_cybermetric_500.jsonl")
METADATA_PATH = os.path.join(DATA_DIR, "master_metadata.json")

In [ ]:
SEED = 3407
VAL_FRACTION = 0.05

In [ ]:
N_FENRIR = 35000
N_CVE = 12000
N_CYBERMETRIC = None          # the 2000-subset is used whole
PHISHING_UPSAMPLE_FACTOR = 4  # 832 rows is too small to survive the mix otherwise

In [ ]:
CYBERMETRIC_TRAIN_SIZE = "2000"   # options: "80", "500", "2000", "10000"
CYBERMETRIC_HOLDOUT_SIZE = "500"  # never trained on, reserved for honest evaluation

In [ ]:
PHISHING_SOURCE = "luongnv89/phishing-email"

In [ ]:
# Optional fifth source. Off by default: same system-prompt lineage as Fenrir.
# When True, a MinHash near-duplicate pass drops Trendyol rows that echo Fenrir.
INCLUDE_TRENDYOL = False
MINHASH_THRESHOLD = 0.85   # Jaccard similarity above which a row is treated as duplicate
MINHASH_NUM_PERM = 128


In [ ]:
MIN_ASSISTANT_CHARS = 40
MAX_USER_CHARS = 12000
MAX_ASSISTANT_CHARS = 12000

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Output directory: {DATA_DIR}")


Output directory: /content/drive/MyDrive/Aegis-CyberSec-Guard/dataV5


In [ ]:
import json
import random
import hashlib
from collections import Counter
from datetime import datetime, timezone

from datasets import load_dataset, Dataset, concatenate_datasets

In [ ]:
random.seed(SEED)

In [ ]:
# Four Roles

SYS_ANALYST = (
    "You are Aegis-Guard, a blue-team cybersecurity analyst assistant. "
    "You analyze logs, network events, and vulnerability data to detect threats and "
    "recommend defensive remediation. You never generate exploit code, malware, or "
    "offensive attack tooling. Explain your reasoning and cite the relevant MITRE "
    "ATT&CK technique or CVE when applicable."
)

SYS_KNOWLEDGE = (
    "You are Aegis-Guard answering a cybersecurity knowledge question. "
    "State the correct option, then give a one or two sentence justification. "
    "Be concise. Do not write an essay."
)

SYS_VULN = (
    "You are Aegis-Guard acting as a vulnerability analyst. "
    "Given a CVE record, summarize the vulnerability, its affected products, its "
    "severity, and the recommended mitigation. Use a structured, factual style. "
    "Do not speculate beyond the record Scope."
)

SYS_PHISHING = (
    "You are Aegis-Guard acting as an email security analyst. "
    "Given raw email content, classify it as phishing or benign and explain the "
    "indicators you used, citing the specific technique (for example homoglyph "
    "spoofing, credential harvesting, urgency manipulation) where relevant."
)


In [ ]:
# Schema Helpers
#Resolve them case-insensitively instead of hardcoding, so a repo
# renaming a column does not silently produce empty strings.
CANONICAL_COLUMNS = ["messages", "source", "task"]

In [ ]:

def pick(example, *candidates, default=""):
    """Return the first present, non-empty value among candidate column names."""
    lowered = {k.lower(): v for k, v in example.items()}
    if not candidates:
        return default
    for c in candidates:
        v = lowered.get(c.lower())
        if v not in (None, ""):
            return v
    return default


In [ ]:
def unescape_text(text: str) -> str:
    """
    El dataset de origen tiene saltos de linea escapados, a veces doble-escapados.
    Normaliza \\\\n -> \\n -> salto real. El orden importa: primero el doble, luego el simple.
    Tambien maneja \\t y \\r por si aparecen.
    """
    if not isinstance(text, str):
        return text
    text = text.replace("\\\\n", "\n").replace("\\\\t", "\t").replace("\\\\r", "")
    text = text.replace("\\n", "\n").replace("\\t", "\t").replace("\\r", "")
    return text

In [ ]:
def make_record(system, user, assistant, source, task):
    return {
        "messages": [
            {"role": "system", "content": unescape_text(str(system).strip())},
            {"role": "user", "content": unescape_text(str(user).strip())},
            {"role": "assistant", "content": unescape_text(str(assistant).strip())},
        ],
        "source": source,
        "task": task,
    }

In [ ]:
def drop_extra_columns(ds: Dataset) -> Dataset:

    extra = [c for c in ds.column_names if c not in CANONICAL_COLUMNS]
    return ds.remove_columns(extra) if extra else ds


In [ ]:
def is_usable(ex):
    assistant = str(pick(ex, "Assistant", "assistant", "output", "response"))
    if len(assistant) < 200:   # los stubs REJECTED rondan los 344 chars, pero el texto util es mas largo
        return False
    lowered = assistant.lower()
    bad_markers = (
        "no description available",
        "state**: rejected",
        "state**: reserved",
        "state**: disputed",
        "state**: withdrawn",
    )
    return not any(m in lowered for m in bad_markers)

## Source Loaders

In [ ]:

def load_fenrir(n: int = N_FENRIR) -> Dataset:
    """
    AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.1
    Long-form defensive reasoning. Supersedes Heimdall-v1.1 and Cybersecurity-Dataset-v1.
    We override the source system prompt with our own so the analyst role is consistent.
    """
    ds = load_dataset("AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.1", split="train")
    print(f"  fenrir raw: {len(ds)} rows, columns: {ds.column_names}")

    def to_chat(ex):
        return make_record(
            system=SYS_ANALYST,
            user=pick(ex, "user", "instruction", "question", "prompt"),
            assistant=pick(ex, "assistant", "output", "response", "answer"),
            source="fenrir_v2.1",
            task="analyst",
        )

    ds = ds.map(to_chat, remove_columns=ds.column_names)
    ds = drop_extra_columns(ds)

    if n is not None and len(ds) > n:
        ds = ds.shuffle(seed=SEED).select(range(n))
    return ds


In [ ]:
def load_cve(n: int = N_CVE) -> Dataset:
    """
    AlicanKiraz0/All-CVE-Records-Training-Dataset
    ~297k rows, but a large share are REJECTED or placeholder records with no usable
    description. Filter to substantive records first, then subsample. Feeding all 297k
    in would drown every other source.
    """
    ds = load_dataset("AlicanKiraz0/All-CVE-Records-Training-Dataset", split="train")
    print(f"  cve raw: {len(ds)} rows, columns: {ds.column_names}")

    rejected_markers = (
        "** REJECT **",
        "DO NOT USE THIS CANDIDATE NUMBER",
        "This candidate has been withdrawn",
        "** DISPUTED **",
        "** RESERVED **",
    )

    def is_usable(ex):
      assistant = str(pick(ex, "Assistant", "assistant", "output", "response"))
      if len(assistant) < 200:
          return False
      lowered = assistant.lower()
      bad_markers = (
          "no description available",
          "state**: rejected",
          "state**: reserved",
          "state**: disputed",
          "state**: withdrawn",
      )
      return not any(m in lowered for m in bad_markers)

    before = len(ds)
    ds = ds.filter(is_usable)
    print(f"  cve after filtering rejected/empty: {len(ds)} (dropped {before - len(ds)})")

    def to_chat(ex):
        return make_record(
            system=SYS_VULN,
            user=pick(ex, "user", "instruction", "question", "prompt"),
            assistant=pick(ex, "assistant", "output", "response", "answer"),
            source="all_cve_records",
            task="vuln_analysis",
        )

    ds = ds.map(to_chat, remove_columns=ds.column_names)
    ds = drop_extra_columns(ds)

    if n is not None and len(ds) > n:
        ds = ds.shuffle(seed=SEED).select(range(n))
    return ds



In [ ]:
def load_heimdall(n=None):
    """
    AlicanKiraz0/Cybersecurity-Dataset-Heimdall-v1.1
    Same author/lineage as Fenrir. Included with an exact-dedup safety net:
    the Cell 12 dedup on user-hash will drop any question already present in Fenrir.
    """
    ds = load_dataset("AlicanKiraz0/Cybersecurity-Dataset-Heimdall-v1.1", split="train")
    print(f"  heimdall raw: {len(ds)} rows, columns: {ds.column_names}")

    def to_chat(ex):
        return make_record(
            system=SYS_ANALYST,
            user=pick(ex, "user", "instruction", "question", "prompt"),
            assistant=pick(ex, "assistant", "output", "response", "answer"),
            source="heimdall_v1.1",
            task="analyst",
        )

    ds = ds.map(to_chat, remove_columns=ds.column_names)
    ds = drop_extra_columns(ds)
    if n is not None and len(ds) > n:
        ds = ds.shuffle(seed=SEED).select(range(n))
    return ds

In [ ]:
def load_phishing(source: str = PHISHING_SOURCE, upsample: int = PHISHING_UPSAMPLE_FACTOR) -> Dataset:
    """
    Phishing triage folded into the same model as a second task.
    832 rows against 35k Fenrir rows would be statistically invisible, so upsample.
    """
    ds = load_dataset(source, split="train")
    print(f"  phishing raw: {len(ds)} rows, columns: {ds.column_names}")

    def to_chat(ex):
        user_content = ""
        assistant_content = ""
        # The phishing dataset stores messages in a 'conversations' list
        if 'conversations' in ex and isinstance(ex['conversations'], list):
            for message in ex['conversations']:
                if message.get('from') == 'human':
                    user_content = message.get('value', '')
                elif message.get('from') in ('assistant', 'gpt'):
                    assistant_content = message.get('value', '')

        # Fallback to 'label' if assistant content is not found or is empty from conversations
        if not assistant_content and 'label' in ex:
             assistant_content = str(ex['label'])

        return make_record(
            system=SYS_PHISHING,
            user=user_content,
            assistant=assistant_content,
            source="phishing_email",
            task="phishing",
        )

    ds = ds.map(to_chat, remove_columns=ds.column_names)
    ds = drop_extra_columns(ds)


    return ds

In [ ]:

def load_trendyol() -> Dataset:
    """
    Optional. Same system-prompt lineage as Fenrir, so it is off by default.
    Enable INCLUDE_TRENDYOL to pull it in and let the MinHash pass in Cell 11
    decide empirically how much genuinely new content it adds.
    """
    ds = load_dataset("Trendyol/Trendyol-Cybersecurity-Instruction-Tuning-Dataset", split="train")
    print(f"  trendyol raw: {len(ds)} rows, columns: {ds.column_names}")

    def to_chat(ex):
        return make_record(
            system=SYS_ANALYST,
            user=pick(ex, "user", "instruction", "question", "prompt"),
            assistant=pick(ex, "assistant", "output", "response", "answer"),
            source="trendyol",
            task="analyst",
        )

    ds = ds.map(to_chat, remove_columns=ds.column_names)
    return drop_extra_columns(ds)


In [ ]:
# Load Everything and be Mindfull Attentive about outputs

print("Loading sources...")
sources = {
    "fenrir": load_fenrir(),
    "cve": load_cve(),
    "heimdall": load_heimdall(),
    "phishing": load_phishing(),
}


sources["trendyol"] = load_trendyol()


print("\nPost-load counts:")
for name, ds in sources.items():
    print(f"  {name}: {len(ds)}")



Loading sources...


README.md:   0%|          | 0.00/6.91k [00:00<?, ?B/s]

220426-CyberSec-Dataset_escaped.jsonl:   0%|          | 0.00/431M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

  fenrir raw: 99870 rows, columns: ['system', 'user', 'assistant']


...

CyberSec-Dataset_escaped.jsonl:   0%|          | 0.00/195M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

  trendyol raw: 53201 rows, columns: ['system', 'user', 'assistant']


Map:   0%|          | 0/53201 [00:00<?, ? examples/s]


Post-load counts:
  fenrir: 35000
  cve: 12000
  heimdall: 21257
  phishing: 832
  trendyol: 53201


In [ ]:
fenrir_full = load_fenrir(n=None)   # carga completo, sin submuestrear
fenrir_full = fenrir_full.shuffle(seed=SEED)

HOLDOUT_N = 500
holdout_ds = fenrir_full.select(range(HOLDOUT_N))
fenrir_train = fenrir_full.select(range(HOLDOUT_N, HOLDOUT_N + N_FENRIR))

sources = {
    "fenrir": fenrir_train,
    "cve": load_cve(),
    "phishing": load_phishing(),
    "heimdall": load_heimdall(),
}

  fenrir raw: 99870 rows, columns: ['system', 'user', 'assistant']
  cve raw: 297441 rows, columns: ['System', 'User', 'Assistant']
  cve after filtering rejected/empty: 282054 (dropped 15387)
  phishing raw: 832 rows, columns: ['conversations', 'label', 'metadata']
  heimdall raw: 21257 rows, columns: ['system', 'user', 'assistant']


In [ ]:
# Inspect one sample per source
# Verify the column resolution actually landed before spending time downstream.
# An empty user or assistant field here means a `pick` candidate list needs updating.


In [ ]:

for name, ds in sources.items():
    print(f"===== {name} =====")
    rec = ds[0]
    for msg in rec["messages"]:
        content = msg["content"]
        preview = content[:400] + ("..." if len(content) > 400 else "")
        print(f"--- {msg['role']} ({len(content)} chars) ---")
        print(preview)
    print(f"task={rec['task']} source={rec['source']}")
    print()


===== fenrir =====
--- system (340 chars) ---
You are Aegis-Guard, a blue-team cybersecurity analyst assistant. You analyze logs, network events, and vulnerability data to detect threats and recommend defensive remediation. You never generate exploit code, malware, or offensive attack tooling. Explain your reasoning and cite the relevant MITRE ATT&CK technique or CVE when applicable.
--- user (91 chars) ---
How can investigators identify notification telemetry revealing data exfiltration attempts?
--- assistant (2510 chars) ---
Investigators can identify notification telemetry revealing data exfiltration attempts through systematic analysis of endpoint detection and response (EDR) logs, network traffic metadata, and cloud service alerts. According to NIST SP 800-53, continuous monitoring frameworks should capture behavioral indicators across the MITRE ATT&CK framework's Exfiltration (TA0010) tactic.

**Endpoint Telemetry...
task=analyst source=fenrir_v2.1

===== cve =====
--- system (2

In [ ]:
# Qualty Filters
# Drop malformed rows, empty fileds , stub answrs  wrong lengths
def is_well_formed(rec) -> bool:
    msgs = rec["messages"]
    if len(msgs) != 3:
        return False
    user = msgs[1]["content"]
    assistant = msgs[2]["content"]
    if not user or not assistant:
        return False
    if len(assistant) < MIN_ASSISTANT_CHARS:
        return False
    if len(user) > MAX_USER_CHARS or len(assistant) > MAX_ASSISTANT_CHARS:
        return False
    return True


In [ ]:

filter_report = {}
for name, ds in sources.items():
    before = len(ds)
    # Phishing answers are legitimately short ("phishing" / "benign"), so relax the floor.
    if name == "phishing":
        sources[name] = ds.filter(
            lambda r: len(r["messages"]) == 3
            and r["messages"][1]["content"]
            and r["messages"][2]["content"]
            and len(r["messages"][1]["content"]) <= MAX_USER_CHARS
        )
    else:
        sources[name] = ds.filter(is_well_formed)
    after = len(sources[name])
    filter_report[name] = {"before": before, "after": after, "dropped": before - after}
    print(f"{name}: {before} -> {after} (dropped {before - after})")


Filter:   0%|          | 0/35000 [00:00<?, ? examples/s]

fenrir: 35000 -> 34994 (dropped 6)


Filter:   0%|          | 0/12000 [00:00<?, ? examples/s]

cve: 12000 -> 11995 (dropped 5)


Filter:   0%|          | 0/832 [00:00<?, ? examples/s]

phishing: 832 -> 825 (dropped 7)


Filter:   0%|          | 0/21257 [00:00<?, ? examples/s]

heimdall: 21257 -> 21253 (dropped 4)


In [ ]:
#  Exact deduplication
# Hash on the user turn. Two rows asking the identical question teach nothing twice.

def user_hash(rec) -> str:
    return hashlib.sha256(rec["messages"][1]["content"].strip().lower().encode("utf-8")).hexdigest()


combined = concatenate_datasets(list(sources.values()))
print(f"Combined before exact dedup: {len(combined)}")

seen = set()
keep_indices = []
for i, rec in enumerate(combined):
    h = user_hash(rec)
    if h not in seen:
        seen.add(h)
        keep_indices.append(i)

combined = combined.select(keep_indices)
exact_dedup_removed = len(seen) and (len(keep_indices))
print(f"Combined after exact dedup: {len(combined)}")


Combined before exact dedup: 69067
Combined after exact dedup: 57570


In [ ]:
phishing_rows = combined.filter(lambda r: r["task"] == "phishing")
other_rows = combined.filter(lambda r: r["task"] != "phishing")

Filter:   0%|          | 0/57570 [00:00<?, ? examples/s]

Filter:   0%|          | 0/57570 [00:00<?, ? examples/s]

In [ ]:
phishing_up = concatenate_datasets([phishing_rows] * 4)
combined = concatenate_datasets([other_rows, phishing_up])
print(f"phishing unique: {len(phishing_rows)} -> upsampled: {len(phishing_up)}")

phishing unique: 367 -> upsampled: 1468


In [ ]:
#  Holdout contamination scrub
# Remove any training row whose question also appears in the evaluation holdout.
# This runs after dedup so it operates on the final training pool.

holdout_hashes = {user_hash(r) for r in holdout_ds}

before = len(combined)
combined = combined.filter(lambda r: user_hash(r) not in holdout_hashes)
contamination_removed = before - len(combined)

print(f"Holdout contamination scrub: removed {contamination_removed} rows")
print(f"Training pool: {len(combined)}")


Filter:   0%|          | 0/58671 [00:00<?, ? examples/s]

Holdout contamination scrub: removed 170 rows
Training pool: 58501


In [ ]:
# Optional MinHash near-duplicate pass
# Only meaningful when INCLUDE_TRENDYOL is True. Exact hashing catches identical
# questions; MinHash catches paraphrases from the same generation template.

if INCLUDE_TRENDYOL:
    from datasketch import MinHash, MinHashLSH

    def shingles(text, k=5):
        tokens = text.lower().split()
        if len(tokens) < k:
            return {" ".join(tokens)}
        return {" ".join(tokens[i:i + k]) for i in range(len(tokens) - k + 1)}

    def build_minhash(text):
        m = MinHash(num_perm=MINHASH_NUM_PERM)
        for s in shingles(text):
            m.update(s.encode("utf-8"))
        return m

    lsh = MinHashLSH(threshold=MINHASH_THRESHOLD, num_perm=MINHASH_NUM_PERM)
    keep = []
    near_dupes = 0

    for i, rec in enumerate(combined):
        m = build_minhash(rec["messages"][1]["content"])
        if lsh.query(m):
            near_dupes += 1
            continue
        lsh.insert(str(i), m)
        keep.append(i)

    before = len(combined)
    combined = combined.select(keep)
    print(f"MinHash near-duplicate pass: {before} -> {len(combined)} (removed {near_dupes})")
else:
    near_dupes = 0
    print("MinHash pass skipped (INCLUDE_TRENDYOL is False).")


MinHash pass skipped (INCLUDE_TRENDYOL is False).


In [ ]:
raw_cve = load_dataset("AlicanKiraz0/All-CVE-Records-Training-Dataset", split="train")
ex = raw_cve[0]
print("Keys:", ex.keys())
print("Assistant field length:", len(str(ex.get("Assistant", ""))))
print("First 500 chars of Assistant:")
print(str(ex.get("Assistant", ""))[:500])
# cuenta cuantos son REJECTED en una muestra
sample = raw_cve.select(range(5000))
rejected = sum(1 for e in sample if "no description available" in str(e.get("Assistant","")).lower())
print(f"REJECTED-like in first 5000: {rejected} ({100*rejected/5000:.1f}%)")

Keys: dict_keys(['System', 'User', 'Assistant'])
Assistant field length: 1559
First 500 chars of Assistant:
## CVE-2010-3763 Vulnerability Details

### CVE Metadata
- **CVE ID**: CVE-2010-3763
- **State**: PUBLISHED
- **Published Date**: October 05, 2010 at 21:00 UTC
- **Last Updated**: August 07, 2024 at 03:18 UTC
- **Reserved Date**: October 05, 2010 at 00:00 UTC
- **Assigned By**: mitre

### Vulnerability Description
Cross-site scripting (XSS) vulnerability in core/summary_api.php in MantisBT before 1.2.3 allows remote attackers to inject arbitrary web script or HTML via the Summary field, a differ
REJECTED-like in first 5000: 289 (5.8%)


In [ ]:
# Composition report
# Read this before you accept the dataset. If one source dominates beyond intent,
# adjust the N_* targets in Cell 4 and re-run rather than discovering it mid-training.

source_counts = Counter(combined["source"])
task_counts = Counter(combined["task"])
total = len(combined)

print(f"Total training rows: {total}\n")

print("By source:")
for s, c in source_counts.most_common():
    print(f"  {s:<28} {c:>7}  ({100 * c / total:5.2f}%)")

print("\nBy task:")
for t, c in task_counts.most_common():
    print(f"  {t:<28} {c:>7}  ({100 * c / total:5.2f}%)")

assistant_lens = [len(r["messages"][2]["content"]) for r in combined]
user_lens = [len(r["messages"][1]["content"]) for r in combined]

def describe(name, values):
    values = sorted(values)
    n = len(values)
    print(
        f"  {name:<12} min={values[0]:>6}  p50={values[n // 2]:>6}  "
        f"p95={values[int(n * 0.95)]:>6}  max={values[-1]:>6}"
    )

print("\nCharacter lengths:")
describe("user", user_lens)
describe("assistant", assistant_lens)


Total training rows: 58501

By source:
  fenrir_v2.1                    33772  (57.73%)
  all_cve_records                11989  (20.49%)
  heimdall_v1.1                  11272  (19.27%)
  phishing_email                  1468  ( 2.51%)

By task:
  analyst                        45044  (77.00%)
  vuln_analysis                  11989  (20.49%)
  phishing                        1468  ( 2.51%)

Character lengths:
  user         min=    40  p50=   139  p95=   224  max=  9818
  assistant    min=    48  p50=  2408  p95=  5209  max= 11303


In [ ]:
# Train/validation split

split = combined.shuffle(seed=SEED).train_test_split(test_size=VAL_FRACTION, seed=SEED)
train_ds, val_ds = split["train"], split["test"]

print(f"train: {len(train_ds)}")
print(f"val:   {len(val_ds)}")



train: 55575
val:   2926


In [ ]:
# Write to Drive


def write_jsonl(ds: Dataset, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for rec in ds:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    size_mb = os.path.getsize(path) / 1e6
    print(f"Wrote {len(ds):>7} rows -> {path}  ({size_mb:.1f} MB)")


In [ ]:
write_jsonl(train_ds, TRAIN_PATH)
write_jsonl(val_ds, VAL_PATH)


Wrote   55575 rows -> /content/drive/MyDrive/Aegis-CyberSec-Guard/dataV5/master_train.jsonl  (176.7 MB)
Wrote    2926 rows -> /content/drive/MyDrive/Aegis-CyberSec-Guard/dataV5/master_val.jsonl  (9.4 MB)


In [ ]:
#  Metadata sidecar
# Provenance matters. Six months from now you will not remember which subsample of CVE
# records went in, and neither will anyone reading the repo.

metadata = {
    "dataset_name": "aegis-cybersec-guard-master",
    "version": "1.0",
    "built_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "target_model": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    "sources_included": {
        "fenrir_v2.1": {
            "repo": "AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.1",
            "role": "core defensive reasoning",
            "target_n": N_FENRIR,
        },
        "all_cve_records": {
            "repo": "AlicanKiraz0/All-CVE-Records-Training-Dataset",
            "role": "structured vulnerability analysis",
            "target_n": N_CVE,
            "filters": "dropped REJECTED/DISPUTED/RESERVED and stub descriptions",
        },
        "cybermetric": {
            "repo": "tihanyin/CyberMetric",
            "subset": CYBERMETRIC_TRAIN_SIZE,
            "role": "short-answer factual knowledge",
        },
        "phishing_email": {
            "repo": PHISHING_SOURCE,
            "role": "email triage, second task",
            "upsample_factor": PHISHING_UPSAMPLE_FACTOR,
        },
    },
    "trendyol_included": INCLUDE_TRENDYOL,
    "sources_excluded": {
        "AlicanKiraz0/Cybersecurity-Dataset-Heimdall-v1.1": "superseded by Fenrir v2.1, same lineage",
        "AlicanKiraz0/Cybersecurity-Dataset-v1": "superseded by Fenrir v2.1, same lineage",
        "Trendyol/Trendyol-Cybersecurity-Instruction-Tuning-Dataset": (
            "shares system-prompt lineage with Fenrir; excluded unless INCLUDE_TRENDYOL"
        ),
        "tuandunghcmut/combine-llm-security-benchmark": (
            "gated, access pending; content covered by CyberMetric plus All-CVE"
        ),
    },
    "system_prompts": {
        "analyst": SYS_ANALYST,
        "knowledge": SYS_KNOWLEDGE,
        "vuln_analysis": SYS_VULN,
        "phishing": SYS_PHISHING,
    },
    "pipeline": [
        "load and normalize to messages schema",
        "per-source quality filters",
        "exact dedup on user turn (sha256)",
        "holdout contamination scrub",
        "optional MinHash near-duplicate pass",
        "shuffle and split",
    ],
    "filter_report": filter_report,
    "near_duplicates_removed": near_dupes,

    "composition_by_source": dict(source_counts),
    "composition_by_task": dict(task_counts),
    "counts": {
        "train": len(train_ds),
        "val": len(val_ds),

        "total_before_split": total,
    },
    "evaluation_protocol": (
        "holdout_cybermetric_500.jsonl is never trained on and its questions were "
        "scrubbed from the training pool. Report accuracy on it for uncontaminated metrics."
    ),
    "files": {
        "train": TRAIN_PATH,
        "val": VAL_PATH,
        "holdout": HOLDOUT_PATH,
    },
}

In [ ]:
with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)


In [ ]:
print(f"Metadata written to {METADATA_PATH}\n")
print(json.dumps({k: metadata[k] for k in ("counts", "composition_by_task")}, indent=2))


Metadata written to /content/drive/MyDrive/Aegis-CyberSec-Guard/dataV5/master_metadata.json

{
  "counts": {
    "train": 55575,
    "val": 2926,
    "total_before_split": 58501
  },
  "composition_by_task": {
    "analyst": 45044,
    "vuln_analysis": 11989,
    "phishing": 1468
  }
}


In [ ]:
import json
with open("/content/drive/MyDrive/Aegis-CyberSec-Guard/dataV3/master_train.jsonl") as f:
    for line in f:
        rec = json.loads(line)
        if rec["source"] == "fenrir_v2.1":
            content = rec["messages"][2]["content"]
            print(repr(content[:200]))
            break

"Google Cloud's resource hierarchy—comprising organizations, folders, projects, and resources—creates complex policy inheritance patterns that vary significantly across operational contexts. The NIST C"


In [ ]:
# Reload verification
# Read the files back from Drive exactly as the training script will. If this cell fails,
# the training script would have failed too, but two hours and several credits later.

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    first = json.loads(f.readline())


In [ ]:
assert "messages" in first, "missing messages key"
assert len(first["messages"]) == 3, "expected system/user/assistant triple"
assert first["messages"][0]["role"] == "system"

In [ ]:
line_count = sum(1 for _ in open(TRAIN_PATH, "r", encoding="utf-8"))
assert line_count == len(train_ds), f"line count mismatch: {line_count} vs {len(train_ds)}"


In [ ]:
print("Reload verification passed.")
print(f"  train rows on disk: {line_count}")
print(f"  first record source: {first['source']}, task: {first['task']}")



Reload verification passed.
  train rows on disk: 55575
  first record source: all_cve_records, task: vuln_analysis


In [ ]:
import json
with open("/content/drive/MyDrive/Aegis-CyberSec-Guard/dataV5/master_train.jsonl") as f:
    for line in f:
        rec = json.loads(line)
        if rec["source"] == "fenrir_v2.1":
            content = rec["messages"][2]["content"]
            print(repr(content[:200]))
            break

"Google Cloud's resource hierarchy—comprising organizations, folders, projects, and resources—creates complex policy inheritance patterns that vary significantly across operational contexts. The NIST C"


In [ ]:
raw = load_dataset("luongnv89/phishing-email", split="train")
print(raw[0]["conversations"])
print("label:", raw[0]["label"])

[{'from': 'human', 'value': 'You are an advanced AI security analyst specialized in email threat detection. Analyze the provided email data and determine if it constitutes a phishing attempt.\n\n**Email Data:**\n- **Sender:** Hung Bentley <LontrinitarianWhitney@washingtonpost.com>\n- **Receiver:** user2.14@gvc.ceas-challenge.cc\n- **Date:** Wed, 06 Aug 2008 23:34:45 +0600\n- **Subject:** parvenu bezel synchronous charcoal diffusion\n- **Body:** \ncrumb oughtnt horsetail? trinitarian, crumb bride.\nemblematic lecture trumpery parvenu bezel owly, rival\nglassware realm trinitarian nd horsetail.\n\ndispensary rival blackball\n\nparvenu quint campground? mercurial, schedule bunyan.\nbezel bride trinitarian bezel oughtnt cufflink, trinitarian\nquint dispensary schubert parvenu bunyan.\n\nbride diffusion chairwoman\n\nrealm lecture consumptive? eros, oughtnt blackball.\n\ncankerworm getaway.\n\n\n\n- **Label:** 1\n- **URLs Present:** No\n\n**Analysis Instructions:**\n1. Examine the sender do

In [ ]:
def is_usable(ex):
    assistant = str(pick(ex, "Assistant", "assistant", "output", "response"))
    if len(assistant) < 200:   # los stubs REJECTED rondan los 344 chars, pero el texto util es mas largo
        return False
    lowered = assistant.lower()
    bad_markers = (
        "no description available",
        "state**: rejected",
        "state**: reserved",
        "state**: disputed",
        "state**: withdrawn",
    )
    return not any(m in lowered for m in bad_markers)

In [ ]:
is_usable(raw[0])

False

In [ ]:
# Create HoldOut

In [3]:
import json, random
random.seed(3407)

In [4]:
DATA_DIR = "/content/drive/MyDrive/Aegis-CyberSec-Guard/dataV5"
train_path = f"{DATA_DIR}/master_train.jsonl"

In [5]:
with open(train_path) as f:
    rows = [json.loads(l) for l in f]

In [7]:
random.shuffle(rows)
holdout = rows[:300]
train = rows[300:]

In [8]:
with open(f"{DATA_DIR}/holdout_300.jsonl", "w") as f:
    for r in holdout:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [9]:

with open(train_path, "w") as f:
    for r in train:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [10]:
print(f"train: {len(train)} | holdout: {len(holdout)}")

train: 55275 | holdout: 300
